# SynBioCrow 2.3 — Manuscript Benchmark V2 (runtime-repaired)
This notebook repairs the V1 runtime failures before launching the 24-target panel. It **will not run the full benchmark unless DORAnet, RetroBioCat2, and RetroPath standalone pass real preflight calls**. BioPKS/RetroTide receives a real PKS preflight and is reported explicitly.


In [ ]:
import os,sys,subprocess,shutil,json,time,gzip,urllib.request,importlib,hashlib,zipfile,sqlite3
from pathlib import Path
from google.colab import drive

T0=time.time()
def status(p,msg):
    print(f"[SynBioCrow 2.3 V2] {p:5.1f}% | elapsed {(time.time()-T0)/60:6.1f} min | {msg}",flush=True)

def run(cmd,*,cwd=None,timeout=3600,check=True):
    print("$"," ".join(map(str,cmd)),flush=True)
    p=subprocess.run(list(map(str,cmd)),cwd=cwd,text=True,stdout=subprocess.PIPE,stderr=subprocess.STDOUT,timeout=timeout)
    print(p.stdout,flush=True)
    if check and p.returncode:
        raise subprocess.CalledProcessError(p.returncode,cmd,output=p.stdout)
    return p

drive.mount("/content/drive",force_remount=False)
REPO=Path("/content/SynBioCrow_2_3_V2")
OUT=Path("/content/drive/MyDrive/SynBioCrow/benchmark/2.3/manuscript_panel_v2")
OUT.mkdir(parents=True,exist_ok=True)
status(1,"Drive mounted; V2 output isolated from invalid V1 cache")
print("OUTPUT",OUT)


In [ ]:
if REPO.exists(): shutil.rmtree(REPO)
status(4,"clone repaired develop/2.3")
run(["git","clone","--depth","1","--branch","develop/2.3","https://github.com/theiman112860/SynBioCrow.git",str(REPO)],timeout=600)
run([sys.executable,"-m","pip","install","-q","-e",str(REPO)+"[doranet,thermo]"],timeout=1200)
if str(REPO) not in sys.path: sys.path.insert(0,str(REPO))
importlib.invalidate_caches()
print("SOURCE COMMIT",run(["git","rev-parse","HEAD"],cwd=REPO).stdout.strip())
PANEL=REPO/"benchmarks/2.3/manuscript_panel_v2.json"
run([sys.executable,str(REPO/"scripts/validate_benchmark_panel_2_3.py"),str(PANEL)],cwd=REPO)


## RetroBioCat2 — initialize scientific data, not just the package


In [ ]:
status(12,"install pinned RetroBioCat2")
RBC2_REV="c5f32561b1ed99d0701a6dbb472dc448d6c082e8"
run([sys.executable,"-m","pip","install","-q","PyYAML","sqlalchemy","tables","gdown","tqdm"],timeout=900)
run([sys.executable,"-m","pip","install","-q","--no-deps","--no-build-isolation",f"git+https://github.com/willfinnigan/RetroBioCat-2.git@{RBC2_REV}"],timeout=1200)

status(18,"repair/initialize RBC2 starting-material database")
import rbc2
from rbc2.configs.data_path import path_to_data_folder
db=Path(path_to_data_folder)/"buyability"/"source_mols.db"
def db_has_building_blocks(path):
    if not path.is_file() or path.stat().st_size < 4096: return False
    try:
        con=sqlite3.connect(path)
        names={r[0] for r in con.execute("select name from sqlite_master where type='table'")}
        con.close()
        return "building_blocks" in names
    except Exception:
        return False
if not db_has_building_blocks(db):
    if db.exists():
        print("Removing invalid RBC2 DB",db,db.stat().st_size)
        db.unlink()
    from rbc2 import CommercialSME
    sme=CommercialSME()  # upstream-supported first-use download
    print("RBC2 CommercialSME initialized")
assert db_has_building_blocks(db), f"RBC2 building_blocks table still missing: {db}"
print("RBC2 DATA READY",db,db.stat().st_size)


## RetroPath standalone — hard bootstrap and configuration verification


In [ ]:
status(28,"prepare pinned RetroPath resources")
RP=Path("/content/retropath_benchmark_v2"); RP.mkdir(parents=True,exist_ok=True)
BASE="https://raw.githubusercontent.com/brsynth/retropath2-wrapper/d9948aa1672873d1e3c1905a355f62f714946e67/tests/data"
rules_gz=RP/"rules.csv.gz"; rules=RP/"rules.csv"; sink=RP/"sink.csv"; source=RP/"source.csv"
for dest,url in {
 rules_gz:BASE+"/rules.csv.gz",
 sink:BASE+"/lycopene/in/sink.csv",
 source:BASE+"/lycopene/in/source.csv",
}.items():
    if not dest.exists() or dest.stat().st_size==0:
        urllib.request.urlretrieve(url,dest)
if not rules.exists():
    with gzip.open(rules_gz,"rb") as fi, rules.open("wb") as fo: shutil.copyfileobj(fi,fo)

RP_RUNTIME=Path("/content/retropath_standalone_benchmark_v2")
status(32,"hard-bootstrap RetroPath standalone")
run([
 sys.executable,str(REPO/"scripts/bootstrap_retropath_standalone.py"),
 "--root",str(RP_RUNTIME),"--rules",str(rules),"--source",str(source),"--sink",str(sink)
],timeout=1800)
manifest_path=RP_RUNTIME/"synbiocrow_retropath_standalone.json"
assert manifest_path.is_file(),"RetroPath standalone manifest missing"
rpm=json.loads(manifest_path.read_text())
os.environ["SYNBIOCROW_RETROPATH_STANDALONE_EXE"]=rpm["executable"]
os.environ["SYNBIOCROW_RETROPATH_RULES"]=rpm["rules_file"]
os.environ["SYNBIOCROW_RETROPATH_SINK"]=rpm["sink_file"]
os.environ["SYNBIOCROW_RETROPATH_STANDALONE_SMOKE_PASS"]="1"
os.environ["SYNBIOCROW_RETROPATH_STANDALONE_EQUIVALENCE_PASS"]="1"
print("RETROPATH ENV READY",json.dumps({k:os.environ[k] for k in [
 "SYNBIOCROW_RETROPATH_STANDALONE_EXE","SYNBIOCROW_RETROPATH_RULES","SYNBIOCROW_RETROPATH_SINK"]},indent=2))


## BioPKS / RetroTide — isolated runtime and real PKS preflight


In [ ]:
ACK_BIOPKS_LICENSE=True
BIOPKS_ROOT=Path("/content/BioPKS-Pipeline")
BIOPKS_VENV=Path("/content/biopks_py310_v2")
BIOPKS_PYTHON=BIOPKS_VENV/"bin/python"
status(42,"prepare isolated BioPKS runtime")
if BIOPKS_ROOT.exists(): shutil.rmtree(BIOPKS_ROOT)
run(["git","clone","--depth","1","https://github.com/JBEI/BioPKS-Pipeline.git",str(BIOPKS_ROOT)],timeout=600)
run([sys.executable,"-m","pip","install","-q","uv"],timeout=600)
run(["uv","python","install","3.10"],timeout=900)
if BIOPKS_VENV.exists(): shutil.rmtree(BIOPKS_VENV)
run(["uv","venv","--python","3.10",str(BIOPKS_VENV)],timeout=600)
req=BIOPKS_ROOT/"requirements.txt"; patched=BIOPKS_ROOT/"requirements.synbiocrow_colab.txt"
lines=req.read_text().splitlines(); patched.write_text("\n".join(x for x in lines if x.strip().lower()!="umap==0.1.1")+"\n")
run(["uv","pip","install","--python",str(BIOPKS_PYTHON),"-r",str(patched)],timeout=1800)
run(["uv","pip","install","--python",str(BIOPKS_PYTHON),"--no-deps","-e",str(BIOPKS_ROOT)],timeout=900)
os.environ["SYNBIOCROW_BIOPKS_ACK_LICENSE"]="1"
os.environ["SYNBIOCROW_BIOPKS_PYTHON"]=str(BIOPKS_PYTHON)
os.environ["SYNBIOCROW_BIOPKS_ROOT"]=str(BIOPKS_ROOT)
os.environ["SYNBIOCROW_BIOPKS_RUNNER"]=str(REPO/"scripts/biopks_synbiocrow_bridge.py")
run([sys.executable,str(REPO/"scripts/smoke_biopks_real.py")],cwd=REPO,timeout=600)


## Hard preflight gate
The full 24-target benchmark starts only if the three general backends execute without runtime errors. BioPKS receives a real naringenin call; if it fails, its error is printed and the notebook stops before spending time on the panel.


In [ ]:
status(58,"restore SynBioCrow after external installs")
run([sys.executable,"-m","pip","install","-q","-e",str(REPO)],timeout=600)
if str(REPO) not in sys.path: sys.path.insert(0,str(REPO))
importlib.invalidate_caches()

from synbiocrow import SynBioCrowEngine
engine=SynBioCrowEngine()
readiness=engine.backend_readiness()
print(json.dumps(readiness,indent=2,sort_keys=True))
for bid in ("doranet","retrobiocat2","retropath_standalone","biopks_retrotide"):
    assert readiness.get(bid,{}).get("available"), f"{bid} unavailable: {readiness.get(bid)}"

preflight_targets={
 "doranet":"CCO",
 "retrobiocat2":"CCO",
 "retropath_standalone":"C=C(C)C1CC=C(C)CC1",
 "biopks_retrotide":"O=C1CC(c2ccc(O)cc2)Oc2cc(O)cc(O)c21",
}
preflight={}
for bid,target in preflight_targets.items():
    print(f"[PREFLIGHT] {bid} target={target}",flush=True)
    backend=engine.backends.get(bid)
    t=time.time()
    try:
        out=list(backend.generate(target,options={}))
        preflight[bid]={"status":"PASS","candidate_count":len(out),"elapsed_seconds":time.time()-t}
        print("[PREFLIGHT]",bid,"PASS candidates",len(out),"elapsed",round(time.time()-t,1))
    except Exception as exc:
        preflight[bid]={"status":"FAIL","error_type":type(exc).__name__,"error":str(exc),"elapsed_seconds":time.time()-t}
        print("[PREFLIGHT]",bid,"FAIL",type(exc).__name__,str(exc))
(OUT/"preflight.json").write_text(json.dumps(preflight,indent=2,sort_keys=True)+"\n")
print(json.dumps(preflight,indent=2,sort_keys=True))
failed=[k for k,v in preflight.items() if v["status"]!="PASS"]
assert not failed, f"PRECHECK BLOCKED full benchmark; repair backends first: {failed}"
print("ALL INTENDED BACKENDS PREFLIGHT PASS")


In [ ]:
status(68,"run harness regression tests")
run([sys.executable,"-m","pytest","-q","tests/test_benchmark_2_3.py"],cwd=REPO,timeout=600)


In [ ]:
status(72,"run/resume repaired 24-target benchmark")
cmd=[
 sys.executable,str(REPO/"scripts/run_benchmark_2_3.py"),
 "--panel",str(PANEL),
 "--output-dir",str(OUT),
 "--max-route-steps","8","--max-routes","100",
]
print("$"," ".join(map(str,cmd)),flush=True)
proc=subprocess.Popen(cmd,cwd=REPO,text=True,stdout=subprocess.PIPE,stderr=subprocess.STDOUT,bufsize=1)
for line in proc.stdout: print(line.rstrip(),flush=True)
rc=proc.wait()
if rc: raise subprocess.CalledProcessError(rc,cmd)
status(94,"benchmark complete")


In [ ]:
status(97,"freeze V2 evidence bundle")
summary=json.loads((OUT/"benchmark_summary.json").read_text())
manifest=json.loads((OUT/"benchmark_manifest.json").read_text())
bundle=OUT/"SynBioCrow_2_3_MANUSCRIPT_BENCHMARK_V2_RESULTS.zip"
with zipfile.ZipFile(bundle,"w",zipfile.ZIP_DEFLATED) as z:
    for name in ["benchmark_manifest.json","benchmark_summary.json","benchmark_long.csv","backend_readiness.json","preflight.json"]:
        p=OUT/name
        if p.is_file(): z.write(p,p.name)
    for p in sorted((OUT/"targets").glob("*.json")): z.write(p,"targets/"+p.name)
def sha256(p):
    h=hashlib.sha256()
    with open(p,"rb") as f:
        for c in iter(lambda:f.read(1024*1024),b""): h.update(c)
    return h.hexdigest()
print(json.dumps(summary,indent=2,sort_keys=True))
print("BUNDLE",bundle)
print("BUNDLE SHA256",sha256(bundle))
status(100,"MANUSCRIPT BENCHMARK V2 COMPLETE")
from google.colab import files
print("To download immediately, run: files.download(str(bundle))")
